In [10]:
from langgraph.graph import StateGraph,START, END
from typing import TypedDict, Literal, Annotated
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
import operator
from dotenv import load_dotenv

In [8]:
load_dotenv() 
generator_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.5, max_tokens=800)
evaluator_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_tokens=500)
optimizer_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3, max_tokens=800)

In [17]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

In [18]:
structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [13]:
class TweetState(TypedDict):

    topic: str
    tweet: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

In [15]:
def generate_tweet(state: TweetState):

    messages = [
        SystemMessage(
            content="You are a witty, clever, and highly relatable Twitter/X influencer."
            ),
        HumanMessage(
            content=f"""
        Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

        Guidelines:
        - Do NOT use a question-answer format.
        - Maximum 280 characters.
        - Use observational humor, irony, sarcasm, memes, or cultural references.
        - Make it relatable and shareable.
        - Keep the language simple and conversational, like everyday English.
        - Focus on punchlines, wit, and funny takes rather than explanations.
        """
        )
    ]
    response = generator_llm.invoke(messages).content

    return {'tweet': response}

In [21]:
def evaluate_tweet(state: TweetState):
    messages = [
    SystemMessage(
        content="You are a ruthless, no-nonsense Twitter critic. You evaluate tweets based on humor, originality, punchiness, virality, and proper tweet format."
    ),
    HumanMessage(
        content=f"""
    Evaluate the following tweet:

    Tweet: "{state['tweet']}"

    Criteria:
    1. Originality – Is this content fresh or overused?  
    2. Humor – Does it genuinely make the reader smile, laugh, or chuckle?  
    3. Punchiness – Is it short, sharp, and scroll-stopping?  
    4. Virality Potential – Would people likely retweet or share it?  
    5. Format – Is it a well-formed tweet? (Not a Q&A, not a setup-punchline joke, under 280 characters)

    Auto-reject if:
    - Written in question-answer format (e.g., "Why did..." or "What happens when...")  
    - Exceeds 280 characters  
    - Reads like a traditional setup-punchline joke  
    - Ends with generic, vague, or deflating lines that weaken the humor  

    ### Respond ONLY in structured format:
    - evaluation: "approved" or "needs_improvement"  
    - feedback: One paragraph explaining strengths and weaknesses
    """
        )
    ]

    response = structured_evaluator_llm.invoke(messages)
    return {'evaluation':response.evaluation, 'feedback': response.feedback}

In [20]:
graph = StateGraph(TweetState)

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

NameError: name 'optimize_tweet' is not defined